# Test notebook for DPD functionality

In [1]:
import warnings
import time

warnings.filterwarnings("ignore")

In [2]:
from flowermd.library import LJChain

start_time = time.perf_counter()
molecules = LJChain(
    num_mols=[10],
    lengths=[50],
    bead_sequence=["_A"],
    bead_mass={"_A": 1.0},
    bond_lengths={"_A-_A": 1.0},
    identify_connections=False,
)
mbuild_time = time.perf_counter()
print("mbuild time:", mbuild_time - start_time)

mbuild time: 0.030023250030353665


In [3]:
molecules.identify_connections

False

## New Random Walk System Class

In [4]:
molecules.dihedral_types

set()

In [5]:
from flowermd.library import RandomWalk
import unyt as u

ref_length = 1.0 * u.Unit("nm")
ref_mass = 1.0 * u.Unit("g/mol")
ref_energy = 1.0 * u.Unit("kcal / mol")
ref_values_dict = {"length": ref_length, "mass": ref_mass, "energy": ref_energy}

system = RandomWalk(
    molecules=molecules,
    density=1.1 * u.Unit("nm**-3"),
    bond_length=1.0,
    buffer=0.5,
    base_units=ref_values_dict,
)

(500, 3)


## DPD Forcefield class

In [6]:
from flowermd.library.forcefields import DPD

In [7]:
A = 250000
gamma = 1500
kT = 1.0
r_cut = 1.01
bond_k = 250000
bond_r0 = 1.0
dpd_ff = DPD(
    A=A, gamma=gamma, kT=kT, r_cut=r_cut, bond_k=bond_k, bond_r0=bond_r0
)

## Forcefield class

In [8]:
from flowermd.library import PhantomWalk

sim = PhantomWalk(
    initial_state=system.hoomd_snapshot,
    forcefield=dpd_ff.hoomd_forces,
    gsd_write_freq=100,
    log_write_freq=50,
    n_steps_dpd=500,
    n_steps_fire=100,
)
end_time = time.perf_counter()
print("Total time:", end_time - start_time)

Initializing simulation state from a gsd.hoomd.Frame.
Step 75 of 500; TPS: 560.08; ETA: 0.0 minutes
Step 150 of 500; TPS: 1029.76; ETA: 0.0 minutes
Step 225 of 500; TPS: 1435.9; ETA: 0.0 minutes
Step 300 of 500; TPS: 1804.92; ETA: 0.0 minutes
Step 375 of 500; TPS: 2147.1; ETA: 0.0 minutes
Step 450 of 500; TPS: 2458.32; ETA: 0.0 minutes
Step 25 of 100; TPS: 31250.0; ETA: 0.0 minutes
The closest particles after DPD + FIRE are:  0.9950000047683716
Total time: 0.41601383400848135


In [9]:
import hoomd

for writer in sim.operations.writers:
    if isinstance(writer, hoomd.write.GSD):
        writer.flush()

In [10]:
## option to use gmso parameterization from FF xml.
from flowermd.library.forcefields import Bead_Spring_DPD

gmso_start = time.perf_counter()
system.apply_forcefield(
    force_field=Bead_Spring_DPD(),
    r_cut=1.01,
    kT=1.0,
    speedup_by_molgraph=True,
    speedup_by_moltag=False,
)
gmso_end = time.perf_counter()
print("gmso ff time:", gmso_end - gmso_start)

gmso ff time: 0.2447224579518661


In [11]:
system.hoomd_forcefield